Analyse du lien entre statut et gravité, sur la durée des séjours

In [1]:
import pandas as pd

#importation des données sous forme de dataframes
MCO2022 = pd.read_csv("SAE/2022/MCO_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
SSR2022 = pd.read_csv("SAE/2022/SSR_2022r.csv", sep=";", encoding="latin-1")

UrgP2022 = pd.read_csv("SAE/2022/URGENCES_P_2022a.csv", sep=";",encoding="latin-1")

FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

#ajout d'une colonne aux données indiuant le statut de chaque établissement
MCO2022 = MCO2022.merge(FINESS,on='FI',how='left')
Urg2022 = Urg2022.merge(FINESS, on='FI', how='left')
SSR2022 = SSR2022.merge(FINESS, on='FI', how='left')

In [2]:
MCO2022.head(3)

,BOR,AN,FI,RS,FI_EJ,LIT_MED,JLI_MED,SEJHC_MED,SEJ0_MED,JOU_MED,...,DNEU,DPED,DOPH,ACTCLI_PM,ACTCLI_SAG,ACTTEC_PM,ACTTEC_DEN,ACTTEC_SAG,ACTTEC_PNM,Statut
0,MCO,2022,010000024,CH DE FLEYRIAT,010780054,270.0,92419.0,14972.0,4042.0,82796.0,...,105.0,105.0,NaN,92931.0,2523.0,63629.0,1204.0,5516.0,12582.0,Public
1,MCO,2022,010000032,CH BUGEY SUD,010780062,66.0,24090.0,4020.0,771.0,21185.0,...,90.0,24.0,NaN,17899.0,1822.0,21383.0,0.0,1676.0,7021.0,Public
2,MCO,2022,010000065,CH DE TREVOUX - MONTPENSIER,010780096,59.0,21535.0,1669.0,9.0,18131.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Public


In [3]:
hospidiag = pd.read_csv("Hospidiag/hd2022.csv", sep=";", encoding='latin-1')
hospidiag.head(3)

,finess,rs,champ_pmsi,taa,cat,taille_MCO,taille_M,taille_C,taille_O,A7,...,RH1,RH2,RH3,RH4,RH5,RH6,RH7,RH8,RH9,RH10
0,010007300,CLINIQUE AMBULATOIRE CENDANEG,OQN,TAA,CLI,T1,M1,C1,NaN,"15,2",...,NaN,"31004,3",53812,NaN,"0,3",NaN,NaN,.z,.z,.z
1,010007987,CH HAUTEVILLE,DGF,TAA,CH,T1,M1,C0,NaN,"27,7",...,NaN,NaN,NaN,"15,1",NaN,NaN,NaN,"7,5","42,8",NaN
2,010008407,CH DU HAUT BUGEY,DGF,TAA,CH,T2,M2,C2,O2,"7,2",...,29.0,"14727,9","54779,5","39,8","1,4","5,5",NaN,NaN,NaN,NaN


In [4]:
# Normalisation du finess 
MCO2022['FI'] = MCO2022['FI'].astype(str).str.zfill(9)
MCO2022['FI_EJ'] = MCO2022['FI_EJ'].astype(str).str.zfill(9)
hospidiag['finess'] = hospidiag['finess'].astype(str).str.zfill(9)

In [5]:
df_hosp_clean = hospidiag[['finess', 'A9']]

df_final = pd.merge(
    MCO2022, 
    df_hosp_clean, 
    left_on='FI_EJ', 
    right_on='finess', 
    how='left'
)

df_final = df_final.drop(columns=['finess'])

In [6]:
df_final.head(5)

,BOR,AN,FI,RS,FI_EJ,LIT_MED,JLI_MED,SEJHC_MED,SEJ0_MED,JOU_MED,...,DPED,DOPH,ACTCLI_PM,ACTCLI_SAG,ACTTEC_PM,ACTTEC_DEN,ACTTEC_SAG,ACTTEC_PNM,Statut,A9
0,MCO,2022,010000024,CH DE FLEYRIAT,010780054,270.0,92419.0,14972.0,4042.0,82796.0,...,105.0,NaN,92931.0,2523.0,63629.0,1204.0,5516.0,12582.0,Public,"11,36"
1,MCO,2022,010000032,CH BUGEY SUD,010780062,66.0,24090.0,4020.0,771.0,21185.0,...,24.0,NaN,17899.0,1822.0,21383.0,0.0,1676.0,7021.0,Public,"18,32"
2,MCO,2022,010000065,CH DE TREVOUX - MONTPENSIER,010780096,59.0,21535.0,1669.0,9.0,18131.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Public,"54,82"
3,MCO,2022,010000099,CH DE MEXIMIEUX,010780120,10.0,3650.0,170.0,1.0,2670.0,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,Public,"59,41"
4,MCO,2022,010000107,CH DE PONT DE VAUX,010780138,15.0,5475.0,177.0,3.0,2259.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Public,"52,54"


In [28]:
import numpy as np
import statsmodels.formula.api as smf

# 1. Filtrer les données pour éviter la division par zéro
# On ne garde que les établissements ayant au moins 1 séjour
df_clean = df_final[df_final['SEJHC_MCO'] > 0].copy()

# 2. Calcul de la durée moyenne du séjour (DMS) et transformation log(1+x)
df_clean['DMS'] = df_clean['JOU_MCO'] / df_clean['SEJHC_MCO']
df_clean['log_DMS'] = np.log1p(df_clean['DMS'])

# 3. Nettoyage strict des valeurs infinies et manquantes
df_clean = df_clean.dropna(subset=['log_DMS', 'Statut', 'A9'])

if df_clean['A9'].dtype == 'object':
    df_clean['A9'] = df_clean['A9'].astype(str).str.replace(',', '.')
df_clean['A9'] = pd.to_numeric(df_clean['A9'], errors='coerce')

# 4. Forcer le statut en variable catégorielle (optimisation de la mémoire et du modèle)
df_clean['Statut'] = df_clean['Statut'].astype('category')

# 5. Lancement de la régression
model = smf.ols('log_DMS ~ Statut * A9', data=df_clean).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                log_DMS   R-squared:                       0.313
Model:                            OLS   Adj. R-squared:                  0.312
Method:                 Least Squares   F-statistic:                     359.9
Date:                Mon, 23 Feb 2026   Prob (F-statistic):           1.95e-66
Time:                        23:12:30   Log-Likelihood:                -423.37
No. Observations:                 793   AIC:                             850.7
Df Residuals:                     791   BIC:                             860.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.8081      0.024     75.296      0.0

In [27]:
df_final[['A9','Statut']].sample(10)

,A9,Statut
14,"45,93",Public
1564,"11,6",Public
203,NaN,Privé non lucratif
213,"13,56",Public
457,"14,78",Public
126,"55,07",Public
1370,NaN,Public
1226,NaN,Privé lucratif
660,"26,9",Public
1443,NaN,Privé lucratif
